# Create a Dataset
Create a dataset of decay curves for ML from an ODE.

In [ ]:
import functools
import itertools
import os
import typing as t

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from scipy import integrate

pio.renderers.default = "svg"

__version__ = "0.0.0"

## Ordinary Differential Equation
Consider a physical phenominon based on the following differential equation.
\begin{align}
\frac{dy(t)}{dt} = - \frac{y(t)}{\tau} + I(t)
\end{align}
where:
- $t$: time  
- $y(t)$: output signal
- $\tau$: time constant
- $I(t)$: input signal


Input a gaussian pulse as $I(t)$.
\begin{align}
I(t) = I_0 \exp{(\frac{-(t-t_0)^2}{\sigma^2})}
\end{align}


In [ ]:
def pulse(i0: float, t0: float, sigma: float) -> t.Callable[[float], float]:
    def f(t: float) -> float:
        return i0 * np.exp(-((t - t0) ** 2) / (sigma**2))

    return f


def dydt(
    tau: float, I: t.Callable[[float], float]
) -> t.Callable[[float, float], float]:
    def f(t: float, y: float) -> float:
        return -y / tau + I(t)

    return f


def solve(
    time: t.Sequence[float], dydt: t.Callable[[float, float], float]
) -> t.Sequence[float]:
    sol = integrate.solve_ivp(
        dydt, t_span=(min(time), max(time)), y0=[0.0], method="Radau", t_eval=time
    )
    y = sol.y[0]
    return y / max(y)


I0 = 1.0
t0 = 0.2
sigma = 0.01
tau = 0.1
time = np.linspace(0.0, 1.0, 480)
input_func = pulse(I0, t0, sigma)
px.line(
    {"time": time, "input": input_func(time), "y": solve(time, dydt(tau, input_func))},
    x="time",
    y=["input", "y"],
).add_vline(t0 - 2 * sigma, line=dict(dash="dash"), annotation=dict(text="-2σ")).show()

In the case of these parameters, it is approximately $t_0-2\sigma$ that the output signal start to rise.

## Create a Dataset
Create decay curves by varying $\tau$, $t_0$ and $y_0$, respectively, where $y_0$ is a value of background.

Now $t_0$ is replaced with $t_0 - k \sigma$, which is the staring time of rising.

In [ ]:
random = np.random.RandomState(np.random.MT19937(0))


def add_tau(data: dict[str, t.Any], tau: float) -> dict[str, t.Any]:
    return dict(tau=round(tau, 4), **data)


def add_t0(data: dict[str, t.Any], t0: float) -> dict[str, t.Any]:
    return dict(t0=round(t0, 4), **data)


def add_y(data: dict[str, t.Any], I0: float, sigma: float) -> dict[str, t.Any]:
    k = 2.0
    t0 = 0.05
    n = int(len(data["t"]) * (data["t0"] - t0) / max(data["t"]))
    y = np.roll(
        solve(data["t"], dydt(data["tau"], pulse(I0, t0 + k * sigma, sigma))), n
    )
    y[:n] = 0
    return dict(y=y, **data)


def add_y0(data: dict[str, t.Any], y0: float, noise_rate: float) -> dict[str, t.Any]:
    y = data["y"] + random.normal(
        scale=max(data["y"]) * noise_rate, size=len(data["y"])
    )
    y = (1.0 - y0) * y + y0
    data = dict(y0=round(y0, 4), **data)
    data.update(y=y)
    return data


def add_id(data: dict[str, t.Any], id: int) -> dict[str, t.Any]:
    return dict(id=id, **data)


I0 = 1.0
sigma = 0.01
tau = np.linspace(0.02, 0.5, 25)
t0 = np.linspace(0.05, 0.25, 21)
y0 = np.linspace(0.05, 0.95, 19)
dataset = [dict(t=time)]
dataset = itertools.starmap(add_tau, itertools.product(dataset, tau))
dataset = itertools.starmap(add_t0, itertools.product(dataset, t0))
dataset = map(functools.partial(add_y, I0=I0, sigma=sigma), dataset)
dataset = filter(lambda x: min(x["y"]) > -1e-6, dataset)
dataset = itertools.starmap(
    functools.partial(add_y0, noise_rate=1e-2), itertools.product(dataset, y0)
)
dataset = itertools.chain(*itertools.repeat(list(dataset), 1))
dataset = itertools.starmap(lambda id, data: add_id(data, id), enumerate(dataset))
df = pd.concat(map(pd.DataFrame, dataset))
print("Number of curves:", len(df) // len(time))
df.describe()

In [ ]:
fig = px.line(
    df.query("tau == 0.1"),
    x="t",
    y="y",
    color="id",
    facet_col="t0",
    facet_col_wrap=3,
).update_layout(showlegend=False)
for i, val in enumerate(range(5, 26)):
    fig.add_vline(
        val * 1e-2,
        row=-(i // 3),
        col=i % 3 + 1,
    )
fig.show(height=1600, width=800)

## Save as a CSV

In [ ]:
filename = f"dataset.csv.gz"
df.to_csv(filename, index=False, compression="gzip")
print(os.path.getsize(filename), "Bytes")

Copyright (c) 2022 Shuhei Nitta